In [ ]:
import pandas as pd
import os

# --- UPDATE THIS PATH IF YOUR DATASET FOLDER IS ELSEWHERE ---
DATA_DIR = "./data"
OUTPUT_DIR = "./outputs"

file_path = os.path.join(OUTPUT_DIR, "dataset_ind_only.xlsx")

df = pd.read_excel(file_path)

Jumlah data: (8190, 15)


,tweet_id,display_name,username,followers,tweet,created_at,lang,user_location,tweet_location,likes,retweets,replies,mentions,reply_to_username,tweet_clean
0,1406970754688717056,Яizal do,afrkml,350301.0,Gejala yg ditimbulkan memang umum dan sudah di...,Mon Jun 21 13:42:22 +0000 2021,in,Dukung kami di sini →,NaN,245,44,4,NaN,afrkml,gejala yg ditimbulkan memang umum dan sudah di...
1,1404644500203471104,DEMA FUSA UIN RIL,demaafusauinril,30.0,[SELAMAT HARI DEMAM BERDARAH DENGUE ASEAN 2021...,Tue Jun 15 03:38:40 +0000 2021,in,NaN,NaN,2,0,0,NaN,NaN,selamat hari demam berdarah dengue asean demam...
2,1409518162362568960,🍒,claudiaars__,230.0,Until this minute masih ga habis pikir bisa2ny...,Mon Jun 28 14:24:52 +0000 2021,in,Jakarta,NaN,0,0,1,NaN,NaN,until this minute masih ga habis pikir bisanya...
3,1406454852889107968,Fer,tabrakajalahh,28.0,Bisa2nya org kena covid pamer di sosmed. Coba ...,Sun Jun 20 03:32:22 +0000 2021,in,NaN,NaN,0,0,0,NaN,NaN,bisanya org kena covid pamer di sosmed coba ka...
4,1404615335433563904,JMKI Jawa Tengah,jmkijateng,10.0,Organisasi Kesehatan Dunia ( WHO ) menuliskan ...,Tue Jun 15 01:42:47 +0000 2021,in,NaN,NaN,0,0,1,NaN,jmkijateng,organisasi kesehatan dunia who menuliskan bahw...


In [3]:
df["region"] = df["tweet_location"]
df.loc[df["region"].isna() | (df["region"] == ""), "region"] = df["user_location"]

# drop kosong
df = df[df["region"].notna()]
df = df[df["region"] != ""]
df["region"] = df["region"].str.lower()

In [4]:
# --- Track geolocation source (for Reviewer 2 Comment #7) ---
df["location_source"] = "none"
df.loc[df["tweet_location"].notna() & (df["tweet_location"] != ""), "location_source"] = "tweet_geotag"
df.loc[(df["location_source"] == "none") & df["user_location"].notna() & (df["user_location"] != ""), "location_source"] = "profile_fallback"

print(df["location_source"].value_counts())

location_source
profile_fallback    5589
tweet_geotag         126
Name: count, dtype: int64


In [5]:
df["location_clean"] = (
    df["region"]
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.strip()
)

In [6]:
# normalisasi lokasi populer
alias_map = {
    "bandarlampung": "bandar lampung",
    "lampung indonesia": "bandar lampung",
    "lampung": "bandar lampung",

    "jaksel": "jakarta",
    "jakbar": "jakarta",
    "jakpus": "jakarta",
    "jakut": "jakarta",
    "jaktim": "jakarta",

    "jogja": "yogyakarta",
    "jogjakarta": "yogyakarta",

    "sby": "surabaya",
    "bdg": "bandung"
}

for alias, city in alias_map.items():
    df["location_clean"] = df["location_clean"].str.replace(
        alias,
        city,
        regex=False
    )

In [ ]:
city_ref = pd.read_csv(os.path.join(DATA_DIR, "daftar-nama-daerah.csv"))

# ambil kota saja (type 2)
city_ref = city_ref[city_ref["type"] == 2]

city_ref["city_clean"] = (
    city_ref["name"]
    .str.lower()
    .str.replace("kabupaten ", "", regex=False)
    .str.replace("kota ", "", regex=False)
    .str.strip()
)

# normalisasi jakarta
city_ref["city_clean"] = city_ref["city_clean"].str.replace(
    r"jakarta.*", "jakarta", regex=True
)

city_list = sorted(
    city_ref["city_clean"].unique().tolist(),
    key=len,
    reverse=True
)

In [8]:
def detect_city(location):
    if pd.isna(location):
        return None
    
    for city in city_list:
        if city in location:
            return city
    
    return None

df["city_clean"] = df["location_clean"].apply(detect_city)

In [9]:
# --- Breakdown by source, among successfully geolocated tweets ---
matched = df[df["city_clean"].notna()]
print("Total geolocated:", len(matched))
print(matched["location_source"].value_counts())
print(matched["location_source"].value_counts(normalize=True) * 100)

Total geolocated: 3327
location_source
profile_fallback    3295
tweet_geotag          32
Name: count, dtype: int64
location_source
profile_fallback    99.038173
tweet_geotag         0.961827
Name: proportion, dtype: float64


In [10]:
# --- Sensitivity check: geotag-only subset ---
df_strict = df[df["location_source"] == "tweet_geotag"].copy()
print("Geotag-only total rows:", len(df_strict))
print("Geotag-only matched to city:", df_strict["city_clean"].notna().sum())
print("Coverage (of geotag-only subset):", round(100 * df_strict["city_clean"].notna().sum() / len(df_strict), 2), "%")

strict_city_dist = df_strict[df_strict["city_clean"].notna()]["city_clean"].value_counts()
print("\nTop 10 cities (geotag-only):")
print(strict_city_dist.head(10))

jakarta_share_strict = strict_city_dist.get("jakarta", 0) / strict_city_dist.sum() * 100
print("\nJakarta share (geotag-only):", round(jakarta_share_strict, 2), "%")

top5_share_strict = strict_city_dist.head(5).sum() / strict_city_dist.sum() * 100
print("Top 5 cities share (geotag-only):", round(top5_share_strict, 2), "%")

Geotag-only total rows: 126
Geotag-only matched to city: 32
Coverage (of geotag-only subset): 25.4 %

Top 10 cities (geotag-only):
city_clean
bogor         7
denpasar      3
balikpapan    2
depok         2
blora         2
palu          2
bekasi        1
rembang       1
metro         1
surabaya      1
Name: count, dtype: int64

Jakarta share (geotag-only): 3.12 %
Top 5 cities share (geotag-only): 50.0 %


In [ ]:
import random
random.seed(42)
sample = matched.sample(n=100, random_state=42)[["location_clean", "city_clean"]]
sample.to_excel(os.path.join(OUTPUT_DIR, "geocoding_spotcheck_sample.xlsx"), index=False)
print("Saved sample of 100 rows to geocoding_spotcheck_sample.xlsx")

Saved sample of 100 rows to geocoding_spotcheck_sample.xlsx


In [22]:
print("Total data:", len(df))
print("Terdeteksi kota:", df["city_clean"].notna().sum())
print("Tidak terdeteksi:", df["city_clean"].isna().sum())

Total data: 5715
Terdeteksi kota: 3327
Tidak terdeteksi: 2388


In [ ]:
# This list directly informed the "granularity gap" finding reported in Section 4.5.1
# (e.g., Jakarta subdistricts like "Pulogadung", colloquial aliases like "Solo")
not_found = (
    df[df["city_clean"].isna()]["location_clean"]
    .value_counts()
    .head(100)
)

print(not_found)

location_clean
indonesia                     602
                               77
t                              51
wherever you are               35
pulo gadung indonesia          25
                             ... 
somewhere over the rainbow      3
biak  papua                     3
httpswwwkalcarecomoutlet        3
cyber                           3
central java indonesia          3
Name: count, Length: 100, dtype: int64


In [24]:
print("Total data:", len(df))

print(
    "Detected:",
    df["city_clean"].notna().sum()
)

print(
    "Coverage:",
    round(
        100 * df["city_clean"].notna().sum() / len(df),
        2
    ),
    "%"
)

Total data: 5715
Detected: 3327
Coverage: 58.22 %


In [25]:
city_count = df["city_clean"].nunique()

print("Jumlah kota unik:", city_count)

Jumlah kota unik: 168


In [26]:
df["city_clean"].value_counts().head(30)

city_clean
jakarta              1411
surabaya              160
bandar lampung        146
yogyakarta            138
bandung               109
denpasar              100
depok                  66
malang                 60
semarang               56
tangerang              49
madiun                 47
bogor                  46
bekasi                 40
serang                 27
makassar               26
tegal                  22
banjarmasin            21
surakarta              20
klaten                 20
pekanbaru              19
medan                  19
blitar                 18
kulon progo            18
cirebon                17
padang                 16
tuban                  16
bantul                 16
rembang                15
gresik                 15
tangerang selatan      14
Name: count, dtype: int64

In [ ]:
output_path = os.path.join(OUTPUT_DIR, "final_structured_dataset.xlsx")

df.to_excel(output_path, index=False)

print("Dataset berhasil disimpan:")
print(output_path)

print("Jumlah data:", len(df))
print("Jumlah kota unik:", df["city_clean"].nunique())
print("Coverage lokasi:",
      round(100 * df["city_clean"].notna().sum() / len(df), 2),
      "%")

Dataset berhasil disimpan:
D:\ribkaadevina\college\8.S2\3. SEM 3\Thesis\Playground\final_structured_dataset.xlsx
Jumlah data: 5715
Jumlah kota unik: 168
Coverage lokasi: 58.22 %
